# 05 - Exploratory data analysis of the synthetic dataset

Course guide: [EDA for synthetic datasets](https://www.apartsin.com/teaching/guides/eda-for-synthetic-datasets.html). It asks eight questions before any model is trained, and each section below answers one:

| # | Guide objective | Section |
|---|---|---|
| 1 | Sanity: is the data technically correct? | 1 |
| 2 | Coverage: is every planned scenario present? | 2 |
| 7 | Bias and balance | 3 |
| - | Box statistics (size, position, small objects) | 4 |
| 4 | Diversity: duplicates, near-duplicates, mode collapse | 6 |
| 3 | Realism: FID, IS, real-vs-synthetic statistics | 7 |
| 6 | Leakage: test photos, shortcuts | 8 |
| 5 | Label quality | 9 |
| 8 | Difficulty (proxies; the baseline-vs-stronger comparison needs trained models, see 03/04) | 9 |

Reads the layout written by `01_generate` and `autolabel.py` under `drive_root` (see README, "Synthetic data format"):
`real/annotations.json` (COCO) + `real/split.json`, `synthetic/images/`, `synthetic/metadata.jsonl`, `synthetic/rejected.jsonl`, `synthetic/autolabel.jsonl`, `synthetic/labels/*.txt` (YOLO).
Tables go to `Results/eda/` (CSV/JSON), figures to `Visuals/eda/` (PNG). Section 10 lists every check with PASS / WARN / FAIL.
Anything optional that is missing (scene tags, attributes, autolabel output) skips only the checks that need it.

In [ ]:
import os, subprocess, sys, yaml
BRANCH = "dev"  # set to your branch name to run a branch
DATA_ROOT = os.environ.get("EDA_DATA")  # None: Google Drive (Colab). Or a local folder that holds real/ and synthetic/

here = os.path.abspath(".")
REPO = next((p for p in (here, os.path.dirname(here)) if os.path.exists(f"{p}/config.yaml")), None)
if REPO is None:  # Colab: same clone pattern as the other notebooks
    REPO = "/content/genai-shelter-validator"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", "https://github.com/AlmogSchwartzberg/genai-shelter-validator.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "fetch", "-q"], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "-q", "origin", BRANCH], check=True)
cfg = yaml.safe_load(open(f"{REPO}/config.yaml"))
if DATA_ROOT is None:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = cfg["drive_root"]
OUT = os.environ.get("EDA_OUT", REPO)
RESULTS, VISUALS = f"{OUT}/Results/eda", f"{OUT}/Visuals/eda"
os.makedirs(RESULTS, exist_ok=True); os.makedirs(VISUALS, exist_ok=True)
try:
    import torchmetrics, torch_fidelity
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchmetrics", "torch-fidelity"], check=True)
print("repo:", REPO, "| data:", DATA_ROOT, "| out:", OUT)

In [ ]:
import json, math, random, hashlib, warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from scipy import stats

warnings.filterwarnings("ignore", category=UserWarning)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = cfg["seed"]; torch.manual_seed(SEED); rng = np.random.default_rng(SEED)
SIZE = cfg["generation"]["image_size"]; MIN_PER_CLASS = cfg["generation"]["images_per_class"]
SMALL_PX = cfg.get("small_object_px", 32)

# colors: Okabe-Ito, color-blind safe. Same entity = same color everywhere.
COL = {"real": "#0072B2", "add_objects": "#E69F00", "scene_edits": "#009E73", "synthetic": "#E69F00",
       "required": "#56B4E9", "lookalike": "#F0E442", "forbidden": "#CC79A7", "bad": "#D55E00", "ink": "#555555"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.axisbelow": True, "figure.dpi": 100, "axes.titlesize": 11, "axes.labelsize": 9})

def savefig(name):
    plt.tight_layout(); plt.savefig(f"{VISUALS}/{name}.png", dpi=150, bbox_inches="tight"); plt.show()

def save_table(df, name):
    df.to_csv(f"{RESULTS}/{name}.csv", index=("index" in (df.index.name or ""))); return df

CHECKS, METRICS = [], {}
def check(section, name, ok, detail="", level="FAIL"):
    """ok True -> PASS, False -> `level` (FAIL or WARN), None -> SKIP (the data the check needs is not there)."""
    CHECKS.append({"section": section, "check": name, "status": "SKIP" if ok is None else "PASS" if ok else level, "detail": detail})
    print(f"[{CHECKS[-1]['status']}] {name}: {detail}")

def iou(a, b):
    iw = max(0, min(a[2], b[2]) - max(a[0], b[0])); ih = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih; union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

## 0. Load the dataset

Class id = row order of `Data/objects_table.csv` (same rule as `autolabel.py`). Synthetic images that count as **used** are those that passed the generation filter *and* got a label from the auto-labeler; the rest only appear in the funnel. If `autolabel.jsonl` does not exist yet, the generation boxes from `metadata.jsonl` stand in for the labels (the notebook says so).

In [ ]:
ROOT = Path(DATA_ROOT)
objects_all = pd.read_csv(f"{REPO}/Data/objects_table.csv").fillna("")
ID2NAME = dict(enumerate(objects_all["name"]))
ROLE = objects_all.set_index("name")["role"].to_dict(); PARENT = objects_all.set_index("name")["lookalike_of"].to_dict()
EXCLUDED = set(cfg["generation"].get("exclude_classes") or [])
CLASSES = [n for n in objects_all["name"] if n not in EXCLUDED]
SIZE_RANGE = {r["name"]: tuple(int(v) for v in __import__("re").findall(r"\d+", r["size_range"])[:2]) for _, r in objects_all.iterrows()}

def jl(p):
    return [json.loads(l) for l in open(p, encoding="utf8") if l.strip()] if p.exists() else []

# ---- real photos: COCO export + split.json
coco = json.load(open(ROOT / "real/annotations.json"))
cat = {c["id"]: c["name"] for c in coco["categories"]}
rim = pd.DataFrame(coco["images"]).rename(columns={"id": "image_id"})
for c in ["source_url", "condition", "room_size", "people", "furniture"]:
    if c not in rim: rim[c] = None
def real_path(f):
    return next((p for p in (ROOT / "real" / f, ROOT / "real/images" / f) if p.exists()), ROOT / "real" / f)
rim["path"] = rim.file_name.map(lambda f: str(real_path(f)))
if (ROOT / "real/split.json").exists():
    split_of = {f: k for k, v in json.load(open(ROOT / "real/split.json")).items() for f in v}
else:  # same recipe as synth.load_split, so the numbers match what generation used
    files = sorted(rim.file_name); random.Random(SEED).shuffle(files)
    n_test = round(len(files) * cfg["split"]["test"]); n_val = round((len(files) - n_test) * cfg["split"]["val_from_train"])
    split_of = {**{f: "test" for f in files[:n_test]}, **{f: "val" for f in files[n_test:n_test + n_val]}, **{f: "train" for f in files[n_test + n_val:]}}
    print("real/split.json not found: split computed here with seed", SEED)
rim["split"] = rim.file_name.map(split_of).fillna("unassigned")
ra = pd.DataFrame(coco["annotations"]); ra["cls"] = ra.category_id.map(cat)
ra[["x0", "y0", "w", "h"]] = pd.DataFrame(ra.bbox.tolist(), index=ra.index)
ra["x1"], ra["y1"] = ra.x0 + ra.w, ra.y0 + ra.h
rbx = ra.merge(rim[["image_id", "file_name", "width", "height"]], on="image_id").rename(columns={"width": "W", "height": "H"})
rbx["set"], rbx["name"], rbx["group"] = "real", rbx.file_name, rbx.file_name.map(split_of).fillna("unassigned")
rbx["raw_oob"] = (rbx.x0 < -1) | (rbx.y0 < -1) | (rbx.x1 > rbx.W + 1) | (rbx.y1 > rbx.H + 1)
rbx["origin"], rbx["gen_iou"], rbx["det_iou"] = None, np.nan, np.nan

# ---- synthetic: metadata (passed generation filter), rejected, autolabel, YOLO labels
meta = jl(ROOT / "synthetic/metadata.jsonl"); rej = jl(ROOT / "synthetic/rejected.jsonl"); auto = jl(ROOT / "synthetic/autolabel.jsonl")
auto_by = {a["image"]: a for a in auto}
HAS_LABELS_DIR = (ROOT / "synthetic/labels").exists() and bool(auto)
rows = []
for stage, recs in (("generation_ok", meta), ("rejected_generation", rej)):
    for m in recs:
        flat = {f"attr_{k}": v for k, v in (m.get("attributes") or {}).items()}
        f = m.get("filter") or {}
        rows.append({"id": m.get("id", Path(m["image"]).stem), "image": m["image"], "stage": stage, "method": m.get("method"), "edit_type": m.get("edit_type"),
                     "source_real": m.get("source_real") or m.get("source"), "prompt": m.get("prompt"), "seed": m.get("seed"), "model": m.get("model"),
                     "box_change": f.get("box_change"), "paint_change": f.get("paint_change"), "reasons": "; ".join(f.get("reasons") or []), **flat})
sim = pd.DataFrame(rows)
meta_boxes = {m["image"]: m.get("boxes") or [] for m in meta}
def status(r):
    if r.stage == "rejected_generation": return "rejected_generation"
    if auto:
        a = auto_by.get(r.image)
        if a is None: return "not_labeled_yet"
        return "used" if a["status"] == "ok" else "rejected_autolabel"
    return "used"
sim["status"] = sim.apply(status, axis=1) if len(sim) else []
sim["autolabel_reason"] = sim.image.map(lambda i: (auto_by.get(i) or {}).get("reason"))
sim["name"] = sim.image.map(lambda i: Path(i).name); sim["path"] = sim.image.map(lambda i: str(ROOT / i))

sb_rows, label_problems = [], Counter()
for r in sim[sim.status == "used"].itertuples():
    with Image.open(r.path) as im: W, H = im.size
    gen = [(b["class"], b["xyxy"], b.get("origin")) for b in meta_boxes[r.image]]
    det = [(b["class"], b["xyxy"], b.get("iou")) for b in (auto_by.get(r.image) or {}).get("boxes", [])]
    lp = ROOT / "synthetic/labels" / (Path(r.image).stem + ".txt")
    if HAS_LABELS_DIR:
        if not lp.exists(): label_problems["ok in autolabel.jsonl but no label file"] += 1; continue
        raw = [l.split() for l in lp.read_text().splitlines() if l.strip()]
        cur = []
        for t in raw:
            cid, xc, yc, w, h = int(t[0]), *[float(v) for v in t[1:5]]
            cur.append((ID2NAME.get(cid, f"id_{cid}"), [(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H], "yolo"))
    else:
        cur = [(c, b, "metadata") for c, b, _ in gen]
    for c, b, src in cur:
        g = max(((iou(b, gb), o) for gc, gb, o in gen if gc == c), default=(np.nan, None))
        d = max(((iou(b, db), di) for dc, db, di in det if dc == c), default=(np.nan, None))
        sb_rows.append({"set": "synthetic", "name": r.name, "cls": c, "x0": b[0], "y0": b[1], "x1": b[2], "y1": b[3], "W": W, "H": H,
                        "group": r.method, "origin": g[1], "gen_iou": g[0], "det_iou": d[1] if d[0] > 0.9 else np.nan, "label_source": src,
                        "raw_oob": b[0] < -1 or b[1] < -1 or b[2] > W + 1 or b[3] > H + 1})
sbx = pd.DataFrame(sb_rows)
if not len(sbx): sbx = pd.DataFrame(columns=["set", "name", "cls", "x0", "y0", "x1", "y1", "W", "H", "group", "origin", "gen_iou", "det_iou", "label_source", "raw_oob"])
sbx["w"], sbx["h"] = sbx.x1 - sbx.x0, sbx.y1 - sbx.y0
BX = pd.concat([rbx[sbx.columns.intersection(rbx.columns)], sbx], ignore_index=True)
for c in ["x0", "y0", "x1", "y1"]: BX[c] = BX[c].astype(float)
BX["x0c"], BX["y0c"] = BX.x0.clip(lower=0), BX.y0.clip(lower=0)
BX["x1c"], BX["y1c"] = np.minimum(BX.x1, BX.W), np.minimum(BX.y1, BX.H)
BX["w"], BX["h"] = BX.x1c - BX.x0c, BX.y1c - BX.y0c
BX["size512"] = np.maximum(BX.w / BX.W, BX.h / BX.H) * 512      # long side at 512 px, as the objects table defines it
BX["cx"], BX["cy"] = (BX.x0c + BX.x1c) / 2 / BX.W, (BX.y0c + BX.y1c) / 2 / BX.H
BX["role"] = BX.cls.map(ROLE)

IM = pd.concat([
    rim.assign(set="real", name=rim.file_name, group=rim.split, status="used", method=None, edit_type=None)[["set", "name", "path", "width", "height", "group", "status", "method", "edit_type", "source_real", "source_url", "condition"] if "source_real" in rim else ["set", "name", "path", "width", "height", "group", "status", "method", "edit_type", "source_url", "condition"]],
    sim.assign(set="synthetic", group=sim.method)[[c for c in ["set", "name", "path", "group", "status", "method", "edit_type", "source_real", "prompt", "seed", "model", "reasons", "autolabel_reason", "stage"] + [c for c in sim.columns if c.startswith("attr_")] if c in sim.columns or c == "set"]],
], ignore_index=True)
USED = IM[IM.status == "used"].reset_index(drop=True)
print(f"real photos: {len(rim)} ({rim.split.value_counts().to_dict()}) | synthetic: {len(sim)} generated, {len(USED[USED.set == 'synthetic'])} used")
print("labels:", "YOLO files written by autolabel.py" if HAS_LABELS_DIR else "generation boxes from metadata.jsonl (no autolabel output yet)")
if label_problems: print("label problems:", dict(label_problems))

## 1. Sanity - is the data technically correct?
Files exist and decode, sizes match, class names are known, boxes are valid, nothing is silently missing. The funnel shows how many generated images survive each filter and why the rest were dropped.

In [ ]:
S = "1 Sanity"
bad_files = []
for r in IM.itertuples():
    try:
        with Image.open(r.path) as im: im.verify()
    except Exception as e:
        bad_files.append((r.name, str(e)[:60]))
check(S, "all image files exist and decode", not bad_files, f"{len(bad_files)} bad of {len(IM)}: {bad_files[:3]}")

rw = IM[IM.set == "real"]
decl = rim.set_index("file_name")
mism = [f for f in decl.index if Path(decl.loc[f, "path"]).exists() and Image.open(decl.loc[f, "path"]).size != (decl.loc[f, "width"], decl.loc[f, "height"])]
check(S, "real: COCO width/height match the files", not mism, f"{len(mism)} mismatches {mism[:3]}")

syn_sizes = Counter(Image.open(p).size for p in USED[USED.set == "synthetic"].path)
check(S, f"synthetic: all images are {SIZE}x{SIZE} (PRD 13)", set(syn_sizes) <= {(SIZE, SIZE)}, str(dict(syn_sizes)))
check(S, "synthetic: unique file names", USED[USED.set == "synthetic"].name.is_unique, "")

known = set(objects_all.name)
unk = sorted(set(BX.cls) - known)
check(S, "every class name / id exists in objects_table.csv", not unk, str(unk))
if EXCLUDED: print("excluded classes (config generation.exclude_classes):", sorted(EXCLUDED))

oob = BX[BX.raw_oob]; tiny = BX[(BX.w < 4) | (BX.h < 4)]
check(S, "boxes lie inside the image", len(oob) == 0, f"{len(oob)} of {len(BX)} boxes outside by >1px: {oob.groupby('set').size().to_dict()}")
check(S, "no degenerate boxes (side < 4 px after clipping)", len(tiny) == 0, f"{len(tiny)} boxes", "WARN")
dupbox = BX.round(0).duplicated(["set", "name", "cls", "x0", "y0", "x1", "y1"], keep="first").sum()
check(S, "no duplicated boxes", dupbox == 0, f"{dupbox} repeated box rows", "WARN")

used_names = set(BX.name)
nobox = USED[~USED.name.isin(used_names)]
check(S, "every used image has at least one box", len(nobox) == 0, f"{len(nobox)} images without boxes: {nobox.groupby('set').size().to_dict()} (PRD 10: unlabeled real photos are not used)", "WARN")
unassigned = (rim.split == "unassigned").sum()
check(S, "every real photo is in a split", unassigned == 0, f"{unassigned} photos not in split.json (added after the split was made?)", "WARN")
check(S, "real: source_url on every photo (PRD 3.3)", rim.source_url.notna().all() if rim.source_url.notna().any() else None, f"{rim.source_url.isna().sum()} missing")

for col in ["method", "seed", "prompt", "model", "source_real"]:
    miss = sim[col].isna().sum() if col in sim else len(sim)
    check(S, f"synthetic metadata: '{col}' present", miss == 0, f"{miss} of {len(sim)} missing", "WARN")
b_no_edit = sim[(sim.method == "scene_edits") & sim.edit_type.isna()]
check(S, "scene_edits images have an edit_type", len(b_no_edit) == 0, f"{len(b_no_edit)} without")
if HAS_LABELS_DIR:
    lbl_files = {p.stem for p in (ROOT / "synthetic/labels").glob("*.txt")}
    orphan = lbl_files - {Path(i).stem for i in sim.image}
    rej_with_label = lbl_files & {Path(i).stem for i in sim[sim.status.str.startswith("rejected")].image}
    check(S, "label files and autolabel.jsonl agree", not label_problems and not orphan and not rej_with_label,
          f"{sum(label_problems.values())} ok-without-file, {len(orphan)} files without image, {len(rej_with_label)} files for rejected images")

# funnel: generated -> passed generation filter -> labeled
f = sim.groupby("method").agg(generated=("id", "size"),
        passed_generation=("stage", lambda s: (s == "generation_ok").sum()),
        used=("status", lambda s: (s == "used").sum())).reset_index()
f["lost_generation_filter"] = f.generated - f.passed_generation; f["lost_autolabel"] = f.passed_generation - f.used
save_table(f, "funnel")
reasons = sim[sim.status.str.startswith("rejected")].assign(reason=lambda d: d.reasons.where(d.reasons != "", d.autolabel_reason).fillna("(no reason recorded)").str.replace(r"[\d.]+", "#", regex=True))
rt = reasons.groupby(["method", "status", "reason"]).size().rename("images").reset_index().sort_values("images", ascending=False)
save_table(rt, "rejection_reasons")
display(f); display(rt.head(12))
if len(f):
    fig, ax = plt.subplots(figsize=(6.5, 2.2 + 0.5 * len(f)))
    y = np.arange(len(f))
    ax.barh(y, f.used, color=COL["required"], label="used", height=0.6)
    ax.barh(y, f.lost_autolabel, left=f.used, color="#E69F00", label="rejected by auto-labeler", height=0.6)
    ax.barh(y, f.lost_generation_filter, left=f.used + f.lost_autolabel, color=COL["bad"], label="rejected by generation filter", height=0.6)
    ax.set_yticks(y, f.method); ax.set_xlabel("images"); ax.set_title("Generated images: how many survive the filters"); ax.legend(frameon=False, fontsize=8)
    savefig("funnel")
rate = 1 - f.used.sum() / max(f.generated.sum(), 1)
check(S, "filters do not discard most images", rate < 0.5, f"{rate:.0%} of generated images dropped (PRD plans 25% over-generation)", "WARN")

## 2. Coverage - is every planned scenario present?
Per class, per method, per edit type, per attribute value. PRD thresholds: at least `images_per_class` synthetic images per class (3.7), all four scene-edit types present (3.5), methods A and B of equal size (3.9), and a real test split big enough to say something (3.3).

In [ ]:
S = "2 Coverage"
syn = BX[BX.set == "synthetic"]; real = BX[BX.set == "real"]
ims = USED[USED.set == "synthetic"]
def n_img(df, c): return df[df.cls == c].name.nunique()
cov = pd.DataFrame({"class": CLASSES, "role": [ROLE[c] for c in CLASSES]})
cov["syn_images_any"] = [n_img(syn, c) for c in CLASSES]
cov["syn_images_painted"] = [n_img(syn[(syn.group == "add_objects") & (syn.origin != "source")], c) if (syn.origin == "synthetic").any() else n_img(syn[syn.group == "add_objects"], c) for c in CLASSES]
cov["syn_instances"] = [(syn.cls == c).sum() for c in CLASSES]
for sp in ["train", "val", "test"]:
    cov[f"real_{sp}_images"] = [n_img(real[real.group == sp], c) for c in CLASSES]
cov["real_instances"] = [(real.cls == c).sum() for c in CLASSES]
cov["below_target"] = cov.syn_images_any < MIN_PER_CLASS
save_table(cov, "coverage_per_class"); display(cov)

under = cov[cov.below_target]
check(S, f"every class has >= {MIN_PER_CLASS} synthetic images (PRD 3.7)", under.empty, f"below target: {dict(zip(under['class'], under.syn_images_any))}")
thin = cov[cov.syn_images_painted < MIN_PER_CLASS]
check(S, f"every class has >= {MIN_PER_CLASS} method A images where it is the painted object", thin.empty, f"below: {dict(zip(thin['class'], thin.syn_images_painted))}", "WARN")
no_real = cov[cov.real_instances == 0]["class"].tolist()
check(S, "every class appears in the real photos", not no_real, f"no real instance of {no_real}: synthetic-only classes cannot be validated on real data", "WARN")
no_test = cov[cov.real_test_images == 0]["class"].tolist()
check(S, "every class appears in the real test split", not no_test, f"absent from test: {no_test} (their mAP@50 is undefined)", "WARN")

# method / edit type
mt = ims.groupby("group").size()
et = USED[(USED.set == "synthetic") & (USED.method == "scene_edits")].edit_type.value_counts()
planned_edits = cfg["generation"]["scene_edits"]
missing_edits = [e for e in planned_edits if et.get(e, 0) == 0]
check(S, "all planned scene-edit types are present (PRD 3.5)", not missing_edits, f"{et.to_dict()} missing: {missing_edits}")
if {"add_objects", "scene_edits"} <= set(mt.index):
    ratio = mt["add_objects"] / mt["scene_edits"]
    check(S, "methods A and B have similar size (PRD 3.9 equal amounts)", 0.8 <= ratio <= 1.25, f"A={mt['add_objects']} B={mt['scene_edits']} ratio {ratio:.2f}", "WARN")

# real set vs PRD 3.3
n_real = len(rim); tst = real[real.group == "test"]
test_imgs = rim[rim.split == "test"].file_name
lk = tst[tst.role == "lookalike"].name.nunique(); fb = tst[tst.role == "forbidden"].name.nunique()
check(S, "real: 100-150 photos (PRD 3.3)", 100 <= n_real <= 150, f"{n_real} photos", "WARN")
check(S, "real test: >= 20 photos, >= 10 with a lookalike, >= 5 with a forbidden object (PRD 3.3)", len(test_imgs) >= 20 and lk >= 10 and fb >= 5, f"test={len(test_imgs)} photos, lookalike in {lk}, forbidden in {fb}", "WARN")

# cross tabs
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ct = pd.crosstab(syn.cls, syn.group).reindex(CLASSES).fillna(0)
for ax, (tab, title) in zip(axes, [(ct, "synthetic instances: class x method"), (pd.crosstab(syn.cls, ims.set_index("name").loc[syn.name, "edit_type"].fillna("add_objects").values).reindex(CLASSES).fillna(0), "synthetic instances: class x edit type")]):
    im = ax.imshow(tab.values, cmap="Blues", aspect="auto"); ax.set_title(title); ax.grid(False)
    ax.set_xticks(range(tab.shape[1]), tab.columns, rotation=30, ha="right", fontsize=8); ax.set_yticks(range(len(tab)), tab.index, fontsize=8)
    for i in range(tab.shape[0]):
        for j in range(tab.shape[1]): ax.text(j, i, int(tab.values[i, j]), ha="center", va="center", fontsize=7, color="white" if tab.values[i, j] > tab.values.max() / 2 else "black")
savefig("coverage_class_method")

# attributes and real tags
attr_cols = [c for c in USED.columns if c.startswith("attr_") and USED[c].notna().any()]
cov_attr = []
for c in attr_cols:
    vc = USED[USED.set == "synthetic"][c].value_counts(normalize=True)
    ent = float(-(vc * np.log(vc)).sum() / np.log(len(vc))) if len(vc) > 1 else 0.0
    cov_attr.append({"attribute": c[5:], "distinct_values": len(vc), "top_share": round(vc.iloc[0], 3), "normalized_entropy": round(ent, 3)})
if cov_attr:
    save_table(pd.DataFrame(cov_attr), "coverage_attributes"); display(pd.DataFrame(cov_attr))
    low = [a["attribute"] for a in cov_attr if a["normalized_entropy"] < 0.6]
    check(S, "attribute values are spread out (normalized entropy >= 0.6)", not low, f"concentrated on few values: {low}", "WARN")
else:
    check(S, "attribute coverage", None, "no 'attributes' in metadata.jsonl")
tags = [c for c in ["condition", "room_size", "people", "furniture"] if rim[c].notna().any()]
for c in tags: print(f"real photos by {c}:", rim.groupby(["split", c]).size().unstack(fill_value=0).to_dict("index"))

## 3. Balance - is the distribution skewed?
Compares class frequencies of synthetic and real data (the detector is trained on both, tested on real only), the role mix (required / lookalike / forbidden), lookalike-vs-original ratios and which classes appear together.

In [ ]:
S = "3 Balance"
def share(df): return (df.cls.value_counts(normalize=True).reindex(CLASSES).fillna(0))
sh = pd.DataFrame({"real": share(real), "synthetic": share(syn)})
for m in ["add_objects", "scene_edits"]:
    sh[m] = share(syn[syn.group == m]) if (syn.group == m).any() else 0
tv = 0.5 * (sh.real - sh.synthetic).abs().sum()
METRICS["class_share_total_variation_real_vs_synthetic"] = round(float(tv), 3)
inst = syn.cls.value_counts().reindex(CLASSES).fillna(0)
ratio = inst.max() / max(inst[inst > 0].min(), 1) if (inst > 0).any() else np.nan
METRICS["synthetic_imbalance_ratio_max_over_min"] = round(float(ratio), 2)
save_table(sh.reset_index().rename(columns={"index": "class", "cls": "class"}), "class_share")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [2.2, 1]})
ax = axes[0]; x = np.arange(len(CLASSES)); wd = 0.38
ax.bar(x - wd / 2, sh.real * 100, wd, color=COL["real"], label="real")
ax.bar(x + wd / 2, sh.synthetic * 100, wd, color=COL["synthetic"], label="synthetic")
ax.set_xticks(x, CLASSES, rotation=35, ha="right", fontsize=8); ax.set_ylabel("share of all boxes (%)")
ax.set_title(f"Class balance, real vs synthetic (total variation {tv:.2f})"); ax.legend(frameon=False)
ax = axes[1]
roles = pd.DataFrame({"real": real.role.value_counts(normalize=True), "synthetic": syn.role.value_counts(normalize=True)}).reindex(["required", "lookalike", "forbidden"]).fillna(0)
bottom = np.zeros(2)
for r in roles.index:
    ax.bar(["real", "synthetic"], roles.loc[r].values * 100, bottom=bottom, color=COL[r], label=r, width=0.55, edgecolor="white", linewidth=1.5); bottom += roles.loc[r].values * 100
ax.set_ylabel("% of boxes"); ax.set_title("Role mix"); ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3)
savefig("class_balance")

pairs = [(l, p) for l, p in PARENT.items() if p and l in CLASSES and p in CLASSES]
pr = pd.DataFrame([{"lookalike": l, "original": p, "syn_lookalike": int((syn.cls == l).sum()), "syn_original": int((syn.cls == p).sum()),
                    "real_lookalike": int((real.cls == l).sum()), "real_original": int((real.cls == p).sum())} for l, p in pairs])
if len(pr):
    pr["syn_ratio"] = (pr.syn_lookalike / pr.syn_original.replace(0, np.nan)).round(2); pr["real_ratio"] = (pr.real_lookalike / pr.real_original.replace(0, np.nan)).round(2)
    save_table(pr, "lookalike_pairs"); display(pr)
lopsided = pr[(pr.syn_ratio < 0.5) | (pr.syn_ratio > 2)] if len(pr) else pr
check(S, "each lookalike has a similar count to the object it imitates (0.5x-2x)", lopsided.empty, f"lopsided pairs: {lopsided[['lookalike','syn_ratio']].values.tolist()}", "WARN")
check(S, "no class dominates the synthetic boxes (max/min <= 5)", bool(ratio <= 5), f"max/min = {ratio:.1f}", "WARN")
check(S, "synthetic class mix is close to the real class mix (total variation <= 0.3)", bool(tv <= 0.3), f"TV = {tv:.2f}. Large gaps teach the detector priors the real test set does not have", "WARN")

# objects per image + co-occurrence
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for s, c in [("real", COL["real"]), ("synthetic", COL["synthetic"])]:
    d = BX[BX.set == s].groupby("name").size()
    axes[0].hist(d, bins=np.arange(0.5, max(d.max(), 3) + 1.5), alpha=0.6, color=c, label=f"{s} (mean {d.mean():.1f})", density=True)
axes[0].set_xlabel("boxes per image"); axes[0].set_title("Objects per image"); axes[0].legend(frameon=False)
for ax, (s, d) in zip(axes[1:], [("real", real), ("synthetic", syn)]):
    pres = pd.crosstab(d.name, d.cls).reindex(columns=CLASSES).fillna(0).clip(upper=1)
    co = pres.T @ pres
    ax.imshow(np.log1p(co.values), cmap="Blues"); ax.grid(False); ax.set_title(f"Co-occurrence, {s} (images, log scale)")
    ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=60, ha="right", fontsize=6); ax.set_yticks(range(len(CLASSES)), CLASSES, fontsize=6)
savefig("balance_cooccurrence")

## 4. Box statistics
Size is the long side of the box at 512 px, the unit of `size_range` in the objects table. Compares each class against that table, real against synthetic (two-sample KS test on log size), small-object share (`small_object_px` in the config) and where objects sit in the frame.

In [ ]:
S = "4 Boxes"
BXv = BX[(BX.w >= 1) & (BX.h >= 1)].copy()
BXv["small"] = BXv.size512 < SMALL_PX
BXv["touches_edge"] = (BXv.x0 <= 1) | (BXv.y0 <= 1) | (BXv.x1 >= BXv.W - 1) | (BXv.y1 >= BXv.H - 1)
BXv["aspect"] = BXv.w / BXv.h
rows = []
for c in CLASSES:
    lo, hi = SIZE_RANGE[c]
    for s in ["real", "synthetic"]:
        d = BXv[(BXv.cls == c) & (BXv.set == s)]
        if d.empty: continue
        rows.append({"class": c, "set": s, "n": len(d), "median_size512": round(d.size512.median(), 1), "p10": round(d.size512.quantile(.1), 1), "p90": round(d.size512.quantile(.9), 1),
                     "table_range": f"{lo}-{hi}", "in_table_range": round(((d.size512 >= lo * 0.8) & (d.size512 <= hi * 1.2)).mean(), 2),
                     "median_aspect": round(d.aspect.median(), 2), "small_share": round(d.small.mean(), 2), "edge_share": round(d.touches_edge.mean(), 2)})
bs = pd.DataFrame(rows); save_table(bs, "bbox_stats_per_class"); display(bs)
out_range = bs[(bs.set == "synthetic") & (bs.in_table_range < 0.8)]
check(S, "synthetic boxes follow the size range of the objects table (>= 80% within +-20%)", out_range.empty, f"off-range classes: {dict(zip(out_range['class'], out_range.in_table_range))}", "WARN")

ks = []
for c in CLASSES:
    a = BXv[(BXv.cls == c) & (BXv.set == "real")].size512; b = BXv[(BXv.cls == c) & (BXv.set == "synthetic")].size512
    if len(a) >= 5 and len(b) >= 5:
        r = stats.ks_2samp(np.log(a), np.log(b)); ks.append({"class": c, "n_real": len(a), "n_syn": len(b), "ks": round(r.statistic, 2), "p": round(r.pvalue, 4)})
ks = pd.DataFrame(ks)
if len(ks):
    save_table(ks, "bbox_size_ks_real_vs_synthetic"); display(ks)
    diff = ks[ks.p < 0.05]
    check(S, "synthetic and real box sizes come from similar distributions (KS p >= 0.05 per class)", diff.empty, f"differ: {dict(zip(diff['class'], diff.ks))}. Few real boxes per class, so read as a hint", "WARN")
else:
    check(S, "real vs synthetic box size (KS)", None, "fewer than 5 real or synthetic boxes in every class")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw={"width_ratios": [2.4, 1, 1]})
ax = axes[0]; pos = np.arange(len(CLASSES))
for off, s in [(-0.19, "real"), (0.19, "synthetic")]:
    data = [BXv[(BXv.cls == c) & (BXv.set == s)].size512.values for c in CLASSES]
    keep = [i for i, d in enumerate(data) if len(d)]
    if keep:
        bp = ax.boxplot([data[i] for i in keep], positions=pos[keep] + off, widths=0.32, patch_artist=True, showfliers=False)
        for p in bp["boxes"]: p.set_facecolor(COL[s]); p.set_alpha(0.55)
        for k in ("medians", "whiskers", "caps"):
            for l in bp[k]: l.set_color(COL["ink"])
    ax.plot([], [], color=COL[s], lw=6, alpha=0.6, label=s)
for i, c in enumerate(CLASSES):
    lo, hi = SIZE_RANGE[c]; ax.plot([i - 0.4, i + 0.4], [lo, lo], color=COL["ink"], lw=1, ls="--"); ax.plot([i - 0.4, i + 0.4], [hi, hi], color=COL["ink"], lw=1, ls="--")
ax.axhline(SMALL_PX, color=COL["bad"], lw=0.8); ax.text(len(CLASSES) - 0.5, SMALL_PX, " small", color=COL["bad"], fontsize=7, va="bottom", ha="right")
ax.set_yscale("log"); ax.set_xticks(pos, CLASSES, rotation=35, ha="right", fontsize=8); ax.set_ylabel("long side at 512 px (log)")
ax.set_title("Box size per class (dashed: objects_table range)"); ax.legend(frameon=False, fontsize=8)
for ax, s in zip(axes[1:], ["real", "synthetic"]):
    d = BXv[BXv.set == s]
    ax.hist2d(d.cx, 1 - d.cy, bins=12, range=[[0, 1], [0, 1]], cmap="Blues"); ax.set_title(f"Box centers, {s}"); ax.set_aspect("equal"); ax.grid(False)
savefig("bbox_size_position")

sm = BXv.groupby("set").agg(boxes=("cls", "size"), small_share=("small", "mean"), edge_share=("touches_edge", "mean"), median_size=("size512", "median")).round(3)
display(sm); METRICS["small_object_share"] = sm.small_share.to_dict()

## 5. One decoding pass: pixel statistics, hashes and Inception features
Every used image is read once: resized to 299x299 (squashing the aspect ratio, as the Inception network expects a fixed size), hashed (SHA-1 of the file, plus a 256-bit difference hash for near-duplicates), and summarized (brightness, saturation, sharpness). The Inception pool features (2048-d) are the same ones FID uses; here they also serve as image embeddings for diversity and leakage checks.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class Pass(Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        raw = Path(self.paths[i]).read_bytes()
        im = Image.open(self.paths[i]).convert("RGB")
        g = np.asarray(im.convert("L").resize((17, 16), Image.BILINEAR), dtype=np.int16)
        small = im.resize((299, 299), Image.BILINEAR); a = np.asarray(small, dtype=np.float32)
        lum = a @ np.array([0.299, 0.587, 0.114], dtype=np.float32)
        lap = 4 * lum[1:-1, 1:-1] - lum[:-2, 1:-1] - lum[2:, 1:-1] - lum[1:-1, :-2] - lum[1:-1, 2:]
        hsv = np.asarray(small.convert("HSV"), dtype=np.float32)
        st = np.array([lum.mean(), lum.std(), hsv[..., 1].mean(), lap.var(), *a.mean((0, 1)), len(raw)], dtype=np.float32)
        return torch.from_numpy(np.asarray(small).copy()).permute(2, 0, 1), torch.from_numpy((g[:, 1:] > g[:, :-1]).flatten().astype(np.uint8)), torch.from_numpy(st), hashlib.sha1(raw).hexdigest()

dl = DataLoader(Pass(USED.path.tolist()), batch_size=64, num_workers=2, pin_memory=torch.cuda.is_available(), shuffle=False, drop_last=False)  # drop_last=False: an EDA must see every image
Xs, Hs, Ss, SHA = [], [], [], []
for x, h, s, sha in dl:
    Xs.append(x); Hs.append(h); Ss.append(s); SHA += list(sha)
X = torch.cat(Xs); HB = torch.cat(Hs).float(); ST = torch.cat(Ss).numpy()
USED = USED.assign(sha1=SHA, **dict(zip(["lum_mean", "lum_std", "saturation", "sharpness", "r", "g", "b", "file_bytes"], ST.T)))

from torchmetrics.image.fid import FrechetInceptionDistance
extractor = FrechetInceptionDistance(feature=2048).to(device).inception.eval()
feats = []
with torch.no_grad():
    for i in range(0, len(X), 64): feats.append(extractor(X[i:i + 64].to(device)).detach().float().cpu())
E = torch.nn.functional.normalize(torch.cat(feats), dim=1).numpy()
del extractor
IS_REAL = (USED.set == "real").values; IS_SYN = ~IS_REAL
print("images:", len(USED), "| tensor", tuple(X.shape), "| embeddings", E.shape, "| device", device)

## 6. Diversity - repetitive or mode-collapsed?
Method B edits one real photo several ways, so near-duplicates **of the same source** are expected; the real question is how many different source rooms there are. Near-duplicates are pairs whose 256-bit difference hashes differ in at most `HASH_MAX` bits.

In [ ]:
S = "6 Diversity"
HASH_MAX = 12
Hm = HB @ HB.T; D = (256 - (Hm + (1 - HB) @ (1 - HB).T)).numpy()
iu = np.triu_indices(len(USED), 1)
close = D[iu] <= HASH_MAX
pi, pj = iu[0][close], iu[1][close]
pairs = pd.DataFrame({"a": USED.name.values[pi], "b": USED.name.values[pj], "set_a": USED.set.values[pi], "set_b": USED.set.values[pj], "hash_bits": D[pi, pj],
                      "same_file": np.array(USED.sha1)[pi] == np.array(USED.sha1)[pj],
                      "same_source": [(sa is not None and sa == sb and str(sa) != "nan") for sa, sb in zip(USED.source_real.values[pi], USED.source_real.values[pj])],
                      "method_a": USED.method.values[pi], "method_b": USED.method.values[pj]}) if len(pi) else pd.DataFrame(columns=["a", "b", "set_a", "set_b", "hash_bits", "same_file", "same_source", "method_a", "method_b"])
sp_ = pairs[(pairs.set_a == "synthetic") & (pairs.set_b == "synthetic")]
save_table(pairs.sort_values("hash_bits"), "near_duplicate_pairs")
exact = sp_.same_file.sum(); other = sp_[~sp_.same_file & ~sp_.same_source]
print(f"synthetic pairs: {len(sp_)} near-duplicate ({exact} byte-identical); {int(sp_.same_source.sum())} share a source photo (expected for method B); {len(other)} unrelated near-duplicates")
check(S, "no byte-identical synthetic images", exact == 0, f"{exact} pairs")
check(S, "no near-duplicate synthetic images from different sources", len(other) == 0, f"{len(other)} pairs (hash distance <= {HASH_MAX}/256)", "WARN")
n_syn = int(IS_SYN.sum()); dup_share = len(set(sp_.a) | set(sp_.b)) / max(n_syn, 1)
METRICS["synthetic_images_with_near_duplicate"] = round(dup_share, 3)

# how many distinct rooms?
src = USED[IS_SYN].groupby("method").source_real.agg(images="size", distinct_sources="nunique").reset_index() if USED.source_real.notna().any() else pd.DataFrame()
if len(src):
    src["images_per_source"] = (src.images / src.distinct_sources).round(1); save_table(src, "sources_per_method"); display(src)
    n_train = int((rim.split == "train").sum())
    check(S, "synthetic images are built from many different real photos", src.distinct_sources.min() >= 0.5 * n_train,
          f"distinct sources {dict(zip(src.method, src.distinct_sources))} of {n_train} train photos", "WARN")
else:
    check(S, "distinct source photos", None, "no source_real in metadata")
for c in ["prompt", "seed"]:
    if USED[IS_SYN][c].notna().any():
        u = USED[IS_SYN][c].nunique() / n_syn; check(S, f"unique {c}s (share of images)", u >= 0.5 if c == "prompt" else u >= 0.99, f"{u:.0%} unique", "WARN")

# embeddings: nearest-neighbor similarity and spread
Sim = E @ E.T; np.fill_diagonal(Sim, -1)
def nn(a, b): return Sim[np.ix_(a, b)].max(1)
ri, si = np.where(IS_REAL)[0], np.where(IS_SYN)[0]
nn_rr = nn(ri, ri) if len(ri) > 1 else np.array([]); nn_ss = nn(si, si); nn_sr = nn(si, ri) if len(ri) else np.array([])
def spread(idx): return float(1 - (E[idx] @ E[idx].T)[np.triu_indices(len(idx), 1)].mean()) if len(idx) > 1 else np.nan
div = {"real": spread(ri), "synthetic": spread(si), **{m: spread(np.where(USED.method.values == m)[0]) for m in ["add_objects", "scene_edits"] if (USED.method == m).any()}}
METRICS["mean_pairwise_cosine_distance"] = {k: round(v, 4) for k, v in div.items()}
print("mean pairwise cosine distance:", METRICS["mean_pairwise_cosine_distance"])
check(S, "synthetic images are not less diverse than real ones (spread >= 0.8x real)", bool(div["synthetic"] >= 0.8 * div["real"]) if len(ri) > 1 else None,
      f"spread synthetic {div['synthetic']:.3f} vs real {div['real']:.3f}", "WARN")

from sklearn.cluster import KMeans
k = min(8, max(2, n_syn // 10))
lab = KMeans(k, n_init=10, random_state=SEED).fit_predict(E[si])
pc = np.bincount(lab, minlength=k) / len(lab); ent = float(-(pc[pc > 0] * np.log(pc[pc > 0])).sum() / np.log(k))
METRICS["synthetic_cluster_entropy"] = round(ent, 3)
check(S, f"synthetic images spread over {k} embedding clusters (normalized entropy >= 0.7)", ent >= 0.7, f"entropy {ent:.2f}, largest cluster {pc.max():.0%}", "WARN")

from sklearn.manifold import TSNE
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
ax = axes[0]
for arr, lbl, c in [(nn_rr, "real to real", COL["real"]), (nn_ss, "synthetic to synthetic", COL["synthetic"]), (nn_sr, "synthetic to nearest real", "#999999")]:
    if len(arr): ax.hist(arr, bins=30, alpha=0.55, color=c, label=lbl, density=True)
ax.set_xlabel("cosine similarity of the nearest neighbor (Inception features)"); ax.set_title("Nearest-neighbor similarity: higher = more repetitive"); ax.legend(frameon=False, fontsize=8)
ax = axes[1]
emb2 = TSNE(2, perplexity=max(5, min(30, (len(E) - 1) // 4)), random_state=SEED, init="pca").fit_transform(E)
for key, c, lbl in [("real", COL["real"], "real"), ("add_objects", COL["add_objects"], "synthetic, method A"), ("scene_edits", COL["scene_edits"], "synthetic, method B")]:
    m = (USED.set == "real").values if key == "real" else (USED.method == key).values
    ax.scatter(emb2[m, 0], emb2[m, 1], s=10, alpha=0.7, color=c, label=lbl, linewidths=0)
ax.set_title("t-SNE of Inception features"); ax.legend(frameon=False, fontsize=8); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
savefig("diversity_nn_tsne")

## 7. Realism - does it look like the real photos?
**FID** (`torchmetrics`, 2048-d) and **Inception Score** are diagnostics, not targets (PRD 5). Two caveats are measured here instead of assumed: FID needs thousands of images and we have ~100 real ones, so the number is inflated by sample size. The table therefore also reports the **noise floor** (one random half of the real photos against the other half) and synthetic subsamples of the same size, which are comparable with it. IS is computed against the 1000 ImageNet classes, so it says little about shelters.

In [ ]:
S = "7 Realism"
from torchmetrics.image.inception import InceptionScore
def batches(idx, n=64):
    for i in range(0, len(idx), n): yield X[idx[i:i + n]].to(device)
def fid_between(real_idx, fake_idx):
    if len(real_idx) < 2 or len(fake_idx) < 2: return float("nan")
    m = FrechetInceptionDistance(feature=2048).to(device)
    with torch.no_grad():
        for b in batches(real_idx): m.update(b, real=True)
        for b in batches(fake_idx): m.update(b, real=False)
        return float(m.compute().detach().cpu())
def inception_score(idx):
    if len(idx) < 10: return (float("nan"), float("nan"))
    m = InceptionScore(splits=min(10, len(idx) // 5)).to(device)
    with torch.no_grad():
        for b in batches(idx): m.update(b)
        mean, std = m.compute()
    return float(mean.detach().cpu()), float(std.detach().cpu())

groups = {"synthetic (all)": si, **{f"synthetic, {m}": np.where(USED.method.values == m)[0] for m in ["add_objects", "scene_edits"] if (USED.method == m).any()}}
if len(ri) >= 20:
    half = len(ri) // 2; reps = 3; rows = []
    def sub(idx, n, r): return np.random.default_rng(SEED + r).choice(idx, min(n, len(idx)), replace=False)
    floor, per = [], {g: [] for g in groups}
    for r in range(reps):
        perm = np.random.default_rng(SEED + r).permutation(ri); a, b = perm[:half], perm[half:2 * half]
        floor.append(fid_between(a, b))
        for g, idx in groups.items(): per[g].append(fid_between(a, sub(idx, half, r)))
    rows.append({"comparison": f"real half vs other real half (noise floor, n={half} vs {half})", "FID": np.mean(floor), "std": np.std(floor)})
    for g in groups: rows.append({"comparison": f"{g} vs real half (n={min(half, len(groups[g]))} vs {half})", "FID": np.mean(per[g]), "std": np.std(per[g])})
    rows.append({"comparison": f"synthetic (all, n={len(si)}) vs all real (n={len(ri)}), unequal sizes, biased", "FID": fid_between(ri, si), "std": np.nan})
    fidt = pd.DataFrame(rows).round(2)
    isr = {"real": inception_score(ri), **{g: inception_score(idx) for g, idx in groups.items()}}
    ist = pd.DataFrame([{"set": k, "IS_mean": round(v[0], 3), "IS_std": round(v[1], 3), "n": int((len(ri) if k == "real" else len(groups[k])))} for k, v in isr.items()])
    save_table(fidt, "fid"); save_table(ist, "inception_score"); display(fidt); display(ist)
    METRICS["fid"] = {r["comparison"]: r["FID"] for r in rows}; METRICS["inception_score"] = {k: v[0] for k, v in isr.items()}
    gap = fidt.FID.iloc[1] - fidt.FID.iloc[0]
    check(S, "synthetic FID is within 2x the real-vs-real noise floor", bool(fidt.FID.iloc[1] <= 2 * fidt.FID.iloc[0]) if fidt.FID.iloc[0] > 0 else None,
          f"synthetic {fidt.FID.iloc[1]:.1f} vs floor {fidt.FID.iloc[0]:.1f}. Diagnostic only (PRD 5): with ~{len(ri)} real photos, FID is very noisy", "WARN")
    fig, ax = plt.subplots(figsize=(7.5, 2.6 + 0.35 * len(fidt)))
    sub_t = fidt.iloc[:-1]; colors = [COL["real"]] + [COL["synthetic"] if "all" in c else COL.get(c.split(", ")[1].split(" vs")[0], COL["synthetic"]) for c in sub_t.comparison[1:]]
    ax.barh(range(len(sub_t)), sub_t.FID, xerr=sub_t["std"].fillna(0), color=colors, height=0.6, ecolor=COL["ink"])
    ax.set_yticks(range(len(sub_t)), [c.split(" (")[0] for c in sub_t.comparison], fontsize=8); ax.invert_yaxis(); ax.set_xlabel("FID (lower = closer to real), mean +- std over 3 draws")
    ax.set_title("FID against the noise floor of the real photos"); savefig("fid_vs_noise_floor")
else:
    check(S, "FID / IS", None, f"only {len(ri)} real photos (need >= 20)")
    IS_ = inception_score(si); print("IS (synthetic):", IS_)

# pixel statistics: do the synthetic photos look like the real ones?
stats_cols = {"lum_mean": "brightness", "lum_std": "contrast", "saturation": "saturation", "sharpness": "sharpness (Laplacian variance)"}
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4)); ksr = []
for ax, (c, nm) in zip(axes, stats_cols.items()):
    for s, col in [("real", COL["real"]), ("synthetic", COL["synthetic"])]:
        v = USED[USED.set == s][c]; ax.hist(np.log1p(v) if c == "sharpness" else v, bins=25, alpha=0.55, color=col, label=s, density=True)
    ax.set_title(nm);
    if len(ri) > 4: r = stats.ks_2samp(USED[IS_REAL][c], USED[IS_SYN][c]); ksr.append({"statistic": nm, "ks": round(r.statistic, 2), "p": round(r.pvalue, 4)})
axes[0].legend(frameon=False, fontsize=8); savefig("pixel_statistics")
if ksr:
    ksr = pd.DataFrame(ksr); save_table(ksr, "pixel_statistics_ks"); display(ksr)
    off = ksr[ksr.ks > 0.4]["statistic"].tolist()
    check(S, "image statistics (brightness, contrast, saturation, sharpness) match the real photos (KS <= 0.4)", not off, f"differ strongly: {off}", "WARN")

## 8. Leakage - can a model cheat?
Three kinds: (1) test photos reaching training (PRD 10: generation may only start from `train` photos; no test photo may be duplicated in train), (2) the detector recognizing "synthetic" from low-level traits instead of content, (3) labels readable from file names. The two-sample classifiers use cross-validated AUC: 0.5 = cannot tell real from synthetic, 1.0 = trivial.

In [ ]:
S = "8 Leakage"
split_by_file = rim.set_index("file_name").split.to_dict()
SYN = USED[IS_SYN]
src_files = SYN.source_real.dropna().map(lambda p: Path(str(p)).name)
bad_src = SYN[src_files.map(lambda f: split_by_file.get(f, "unknown") in ("test", "val")).reindex(SYN.index, fill_value=False)]
check(S, "no synthetic image is built from a val or test photo (PRD 10)", bad_src.empty if len(src_files) else None,
      f"{len(bad_src)} images from val/test photos: {bad_src.name.tolist()[:4]}")

real_idx = USED[IS_REAL]
rsplit = real_idx.group.values
cross = pairs[(pairs.set_a == "real") & (pairs.set_b == "real")]
gs = dict(zip(real_idx.name, real_idx.group))
cross = cross[[gs[a] != gs[b] and "test" in (gs[a], gs[b]) for a, b in zip(cross.a, cross.b)]]
check(S, "no real test photo duplicates a train/val photo (PRD 10)", cross.empty, f"{len(cross)} pairs: {cross[['a','b']].values.tolist()[:3]}")

to_test = pairs[(pairs.set_a != pairs.set_b)]
to_test = to_test[[any(gs.get(n) == "test" for n in (a, b)) for a, b in zip(to_test.a, to_test.b)]]
check(S, "no synthetic image is a near-copy of a real test photo", to_test.empty, f"{len(to_test)} pairs")

# shortcut 1: low-level pixel statistics / file traits
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
y = IS_SYN.astype(int)
def auc(F, model):
    if min(y.sum(), (1 - y).sum()) < 10: return float("nan")
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    return float(roc_auc_score(y, cross_val_predict(model, F, y, cv=cv, method="predict_proba")[:, 1]))
pix = USED[["lum_mean", "lum_std", "saturation", "sharpness", "r", "g", "b"]].values
meta_f = np.c_[[Image.open(p).size[0] for p in USED.path], USED.file_bytes.values, USED.path.str.endswith(".png").astype(int)]
res = {"pixel statistics (brightness, color, sharpness)": auc(pix, GradientBoostingClassifier(n_estimators=100, random_state=SEED)),
       "file traits (width, bytes, png)": auc(meta_f, GradientBoostingClassifier(n_estimators=100, random_state=SEED)),
       "Inception embedding (linear probe)": auc(E, make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=2000)))}
sc = pd.DataFrame({"features": list(res), "cv_auc_real_vs_synthetic": [round(v, 3) for v in res.values()]}); save_table(sc, "shortcut_auc"); display(sc)
METRICS["real_vs_synthetic_auc"] = {k: round(v, 3) for k, v in res.items()}
check(S, "pixel statistics alone cannot separate real from synthetic (AUC <= 0.85)", bool(res["pixel statistics (brightness, color, sharpness)"] <= 0.85) if not math.isnan(res["pixel statistics (brightness, color, sharpness)"]) else None,
      f"AUC {res['pixel statistics (brightness, color, sharpness)']:.2f}", "WARN")
fa = res["file traits (width, bytes, png)"]
check(S, "file traits (size, format) do not give away the set", bool(fa <= 0.85) if not math.isnan(fa) else None, f"AUC {fa:.2f}. If high, resize and re-encode real and synthetic images identically before training (PRD: resize before training)", "WARN")
ea = res["Inception embedding (linear probe)"]
check(S, "a linear probe on Inception features does not trivially separate the sets (AUC <= 0.95)", bool(ea <= 0.95) if not math.isnan(ea) else None,
      f"AUC {ea:.2f}. Expected to be high for diffusion images; the detector can use this domain gap as a shortcut when synthetic data are over-represented", "WARN")

# shortcut 2: labels in file names / class only in one set
leak_names = [c for c in CLASSES if USED.name.str.contains(c, regex=False).any()]
check(S, "file names do not reveal the class", not leak_names, f"synthetic names contain the class ({len(leak_names)} classes), real names do not. Harmless unless a loader reads names; export with neutral ids to be safe", "WARN")
only_syn = [c for c in CLASSES if (syn.cls == c).any() and not (real.cls == c).any()]
check(S, "no class exists only in synthetic data", not only_syn, f"synthetic-only classes: {only_syn}: the detector learns them from synthetic style alone", "WARN")

## 9. Label quality and difficulty
Labels come from generation (`origin: synthetic` boxes are where the object was painted; `origin: source` boxes are real annotations carried over) and from the auto-labeler (Grounding-DINO IoU). Checks: how close the detector's confirmation is to its rejection threshold, drift between generation box and final label, contradictory labels, and objects that barely differ from their surroundings (probably not visible, so the label is wrong).
**Difficulty** needs a trained model to be measured honestly (a weak baseline vs a stronger one); that comparison comes from notebooks 03/04. Here only label-free proxies are shown: small objects, crowding and contrast.

In [ ]:
S = "9 Labels & difficulty"
syn_b = BXv[BXv.set == "synthetic"]
if syn_b.det_iou.notna().any():
    thr = cfg.get("autolabel", {}).get("min_iou", 0.4)
    d = syn_b.det_iou.dropna(); near = (d < thr + 0.1).mean()
    METRICS["autolabel_iou_median"] = round(float(d.median()), 3)
    check(S, "painted objects are confirmed well above the auto-label threshold", near <= 0.2, f"{near:.0%} of confirmed boxes within 0.1 of the threshold {thr}; median IoU {d.median():.2f}. Calibrate the threshold on the manual check (PRD 3.4)", "WARN")
else:
    check(S, "auto-label confirmation IoU", None, "no 'iou' in autolabel.jsonl")
if syn_b.gen_iou.notna().any():
    drift = syn_b.gen_iou.dropna(); lo = (drift < 0.9).mean()
    check(S, "final labels equal the generation boxes (IoU >= 0.9)", lo <= 0.05, f"{lo:.0%} differ (autolabel 'refine' on?)", "WARN")

# contradictions: lookalike and its original in the same place; repeated boxes
contra, repeat = [], 0
for name, g in syn_b.groupby("name"):
    rows_ = g.to_dict("records")
    for i, a in enumerate(rows_):
        for b in rows_[i + 1:]:
            ov = iou([a["x0c"], a["y0c"], a["x1c"], a["y1c"]], [b["x0c"], b["y0c"], b["x1c"], b["y1c"]])
            if PARENT.get(a["cls"]) == b["cls"] or PARENT.get(b["cls"]) == a["cls"]:
                if ov > 0.3: contra.append((name, a["cls"], b["cls"], round(ov, 2)))
            elif a["cls"] == b["cls"] and ov > 0.9: repeat += 1
check(S, "no lookalike box overlaps the object it imitates (IoU <= 0.3)", not contra, f"{len(contra)} images: {contra[:3]}")
check(S, "no repeated boxes of the same class (IoU > 0.9)", repeat == 0, f"{repeat} pairs", "WARN")

# visibility: color difference between the inside of a box and a ring around it
def contrast(path, b):
    arr = np.asarray(Image.open(path).convert("RGB"), dtype=np.float32); H, W = arr.shape[:2]
    x0, y0, x1, y1 = [int(round(v)) for v in b]; x0, y0, x1, y1 = max(x0, 0), max(y0, 0), min(x1, W), min(y1, H)
    if x1 - x0 < 2 or y1 - y0 < 2: return np.nan, np.nan
    m = max(4, int(0.25 * max(x1 - x0, y1 - y0))); X0, Y0, X1, Y1 = max(x0 - m, 0), max(y0 - m, 0), min(x1 + m, W), min(y1 + m, H)
    ins = arr[y0:y1, x0:x1]; big = arr[Y0:Y1, X0:X1]; n_in, n_big = ins.shape[0] * ins.shape[1], big.shape[0] * big.shape[1]
    ring = (big.sum((0, 1)) - ins.sum((0, 1))) / max(n_big - n_in, 1)
    return float(np.linalg.norm(ins.mean((0, 1)) - ring) / 255), float(ins.std())
path_of = dict(zip(IM.name + "|" + IM.set, IM.path))
sample = pd.concat([d.sample(min(len(d), 600), random_state=SEED) for _, d in BXv.groupby("set")])
cv = [contrast(path_of[f"{r.name}|{r.set}"], [r.x0, r.y0, r.x1, r.y1]) for r in sample.itertuples()]
sample = sample.assign(contrast=[c[0] for c in cv], inside_std=[c[1] for c in cv])
low = sample[sample.contrast < 0.03].sort_values("contrast")   # mean color inside the box is within 3% of the ring around it
save_table(low[["set", "name", "cls", "group", "x0", "y0", "x1", "y1", "contrast", "inside_std"]], "low_contrast_boxes")
share_low = (low.set == "synthetic").sum() / max((sample.set == "synthetic").sum(), 1)
check(S, "painted objects are visible (box colors differ from the surroundings)", share_low <= 0.05, f"{share_low:.0%} of sampled synthetic boxes have the same mean color as their surroundings; low_contrast_boxes.csv lists them, check by eye (a white sign on a white wall can be fine)", "WARN")

# difficulty proxies
def max_overlap(g):
    r = g[["x0c", "y0c", "x1c", "y1c"]].values
    return max((iou(r[i], r[j]) for i in range(len(r)) for j in range(i + 1, len(r))), default=0.0)
dif = []
for s in ["real", "synthetic"]:
    d = BXv[BXv.set == s]; per_img = d.groupby("name")
    dif.append({"set": s, "images": per_img.ngroups, "boxes_per_image": round(len(d) / max(per_img.ngroups, 1), 2), "small_object_share": round(d.small.mean(), 3),
                "images_with_overlapping_boxes": round((per_img.apply(max_overlap) > 0.05).mean(), 3), "median_contrast": round(sample[sample.set == s].contrast.median(), 3),
                "images_with_lookalike": round(d[d.role == "lookalike"].name.nunique() / max(per_img.ngroups, 1), 3), "images_with_forbidden": round(d[d.role == "forbidden"].name.nunique() / max(per_img.ngroups, 1), 3)})
dif = pd.DataFrame(dif); save_table(dif, "difficulty_proxies"); display(dif)
METRICS["difficulty_proxies"] = dif.set_index("set").to_dict("index")

### Look at the data
Numbers do not replace reading images. The first grid is a random sample (the PRD asks for 16) with boxes colored by role; the second shows the boxes most likely to be wrong (lowest contrast). Review these by hand and record what you find in the slides.

In [ ]:
ROLE_COL = {"required": COL["required"], "lookalike": COL["lookalike"], "forbidden": COL["forbidden"]}
def drawn(name, set_name, px=300):
    path = path_of[f"{name}|{set_name}"]; im = Image.open(path).convert("RGB"); W, H = im.size; im = im.resize((px, px)); d = ImageDraw.Draw(im)
    for r in BX[(BX.set == set_name) & (BX.name == name)].itertuples():
        b = [r.x0c * px / W, r.y0c * px / H, r.x1c * px / W, r.y1c * px / H]; d.rectangle(b, outline=ROLE_COL.get(r.role, "white"), width=3); d.text((b[0] + 3, b[1] + 2), r.cls, fill="white")
    return im
def grid(items, cols, title, fname, titles=None):
    rows = math.ceil(len(items) / cols); fig, axes = plt.subplots(rows, cols, figsize=(2.6 * cols, 2.7 * rows)); axes = np.atleast_1d(axes).ravel()
    for ax in axes: ax.axis("off")
    for i, (n, s) in enumerate(items): axes[i].imshow(drawn(n, s)); axes[i].set_title((titles[i] if titles else f"{s}: {n}")[:34], fontsize=7)
    fig.suptitle(title + "  (light blue: required, yellow: lookalike, pink: forbidden)", fontsize=10); savefig(fname)
samp = lambda s, n, m=None: [(r.name, s) for r in USED[(USED.set == s) & ((USED.method == m) if m else True) & USED.name.isin(set(BX.name))].sample(frac=1, random_state=SEED).head(n).itertuples()]
a_s, b_s, r_s = samp("synthetic", 8, "add_objects"), samp("synthetic", 8, "scene_edits"), samp("real", 8)
grid(a_s + b_s, 4, "Random synthetic samples: method A (top two rows) and method B (bottom two)", "samples_synthetic")
if r_s: grid(r_s, 4, "Random real photos", "samples_real")
if len(low):
    grid([(r.name, r.set) for r in low.head(12).itertuples()], 4, "Boxes most similar to their surroundings (check the label)", "samples_low_contrast", titles=[f"{r.set}: {r.cls}" for r in low.head(12).itertuples()])

## 10. Summary - every check in one table

In [ ]:
summ = pd.DataFrame(CHECKS); save_table(summ, "eda_checks")
json.dump({"metrics": METRICS, "n_real": int(len(rim)), "n_synthetic_used": int(IS_SYN.sum()), "n_synthetic_generated": int(len(sim)),
           "checks": summ.status.value_counts().to_dict()}, open(f"{RESULTS}/eda_metrics.json", "w"), indent=2, default=float)
def color(v): return {"PASS": "background-color:#d4ecdd", "WARN": "background-color:#fbe5b6", "FAIL": "background-color:#f5c6c0", "SKIP": "background-color:#e4e4e4"}.get(v, "")
display(summ.style.map(color, subset=["status"]).set_properties(**{"text-align": "left"}))
print(summ.status.value_counts().to_dict())
fig, ax = plt.subplots(figsize=(7, 2.2))
cnt = summ.status.value_counts().reindex(["PASS", "WARN", "FAIL", "SKIP"]).fillna(0)
ax.barh(["checks"], [cnt["PASS"]], color="#009E73"); left = cnt["PASS"]
for k, c in [("WARN", "#E69F00"), ("FAIL", COL["bad"]), ("SKIP", "#bbbbbb")]:
    ax.barh(["checks"], [cnt[k]], left=left, color=c); left += cnt[k]
for k, x in zip(cnt.index, np.cumsum(cnt.values) - cnt.values / 2):
    if cnt[k]: ax.text(x, 0, f"{k} {int(cnt[k])}", ha="center", va="center", color="white", fontsize=9)
ax.axis("off"); ax.set_title("EDA checks"); savefig("eda_summary")

# Colab: the clone lives in /content and disappears, so keep a copy of the outputs next to the data
if Path(OUT) != Path(DATA_ROOT) and not DATA_ROOT.startswith(OUT):
    import shutil
    for src_dir, name in [(RESULTS, "results/eda"), (VISUALS, "results/eda_figures")]:
        shutil.copytree(src_dir, f"{DATA_ROOT}/{name}", dirs_exist_ok=True)
    print("copy saved to", f"{DATA_ROOT}/results/")
print("Results:", RESULTS, "\nFigures:", VISUALS)